In [19]:
import os
print(os.getcwd())

c:\dev\projects\capstone-quantum-1


In [20]:
import pandas as pd

track_B_df = pd.read_parquet("data/processed/track_B_curated.parquet")

In [21]:
track_B_df = track_B_df.drop(columns=["T10Y3M_delta"])

track_B_df.head()

,UNRATE,PERMIT,S&P 500,UMCSENTx,T10Y3M_level
date,,,,,
1962-04-01,0.0,7.118826,-0.032387,0.0,1.11
1962-05-01,-0.1,7.040536,-0.077267,-4.5,1.18
1962-06-01,0.0,7.050989,-0.124253,0.0,1.18
1962-07-01,-0.1,7.080868,0.023802,0.0,1.09
1962-08-01,0.3,7.090077,0.026844,-3.8,1.16


In [22]:
import numpy as np
import pandas as pd

def generate_poos_splits(
    df,
    target_col,
    sequence_length=24,
    forecast_horizon=1,
    step=1,
    expanding=False,
    return_full_df=False
):
    """
    Pseudo Out-of-Sample (POOS) generator.

    Parameters
    ----------
    df : pd.DataFrame
        Input time series dataframe.
    target_col : str
        Target column name.
    sequence_length : int
        Window size for training.
    forecast_horizon : int
        Forecast horizon (currently supports 1-step usage in most models).
    step : int
        Step size between splits.
    expanding : bool
        If True, uses expanding window instead of rolling.
    return_full_df : bool
        If True, yields full feature dataframe windows.
        If False, yields only target series.
    
    Yields
    ------
    train, test : pd.DataFrame or pd.Series
    """

    n = len(df)

    start = sequence_length

    while start + forecast_horizon <= n:

        if expanding:
            train_idx = slice(0, start)
        else:
            train_idx = slice(start - sequence_length, start)

        test_idx = slice(start, start + forecast_horizon)

        train_block = df.iloc[train_idx]
        test_block = df.iloc[test_idx]

        # --- mode switch ---
        if return_full_df:
            train_out = train_block
            test_out = test_block
        else:
            train_out = train_block[target_col]
            test_out = test_block[target_col]

        yield train_out, test_out

        start += step

In [23]:
import numpy as np
import pandas as pd
from collections import Counter
from pmdarima import auto_arima

# --- parameters ---
target_col = "UNRATE"
sequence_length = 24
forecast_horizon = 1
step = 5
expanding = False

# --- storage ---
y_true_all = []
y_pred_all = []
orders = []

# --- run POOS ---
for train, test in generate_poos_splits(
    track_B_df,
    target_col=target_col,
    sequence_length=sequence_length,
    forecast_horizon=forecast_horizon,
    step=step,
    expanding=expanding
):
    model = auto_arima(
        train,
        seasonal=False,
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore"
    )
    
    forecast = model.predict(n_periods=len(test))
    
    # store predictions
    y_true_all.extend(test.values)
    y_pred_all.extend(forecast)
    
    # store (p,d,q)
    orders.append(model.order)

# --- compute MASE ---
y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)

# numerator: MAE of model
mae = np.mean(np.abs(y_true_all - y_pred_all))

# denominator: MAE of naive (lag-1) forecast on FULL series
y_full = track_B_df[target_col].values
naive_errors = np.abs(y_full[1:] - y_full[:-1])
naive_mae = np.mean(naive_errors)

mase = mae / naive_mae

# --- summarize orders ---
order_counts = Counter(orders)

order_df = pd.DataFrame(
    [(p, d, q, count) for (p, d, q), count in order_counts.items()],
    columns=["p", "d", "q", "count"]
).sort_values("count", ascending=False)

order_summary = {
    "p_mean": np.mean([o[0] for o in orders]),
    "d_mean": np.mean([o[1] for o in orders]),
    "q_mean": np.mean([o[2] for o in orders]),
    "most_common_order": order_counts.most_common(1)[0][0]
}

# --- outputs ---
print(f"MASE: {mase:.4f}")
print("\nMost common (p,d,q):", order_summary["most_common_order"])
print("\nMean orders:", order_summary)
print("\nOrder distribution:")
display(order_df)

MASE: 0.6524

Most common (p,d,q): (0, 0, 0)

Mean orders: {'p_mean': np.float64(0.5973154362416108), 'd_mean': np.float64(0.14093959731543623), 'q_mean': np.float64(0.40268456375838924), 'most_common_order': (0, 0, 0)}

Order distribution:


,p,d,q,count
4,0,0,0,54
0,0,0,1,26
1,1,0,0,26
11,2,0,0,10
9,0,1,1,6
8,0,1,0,5
12,2,0,2,2
10,1,1,0,2
2,1,0,1,2
6,3,0,2,2


In [24]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# --- parameters ---
target_col = "UNRATE"
max_lag = 6
forecast_horizon = 1
step = 1
sequence_length = 24
expanding = False

results = []

# --- POOS loop ---
for train, test in generate_poos_splits(
    df=track_B_df,
    target_col=target_col,
    sequence_length=sequence_length,
    forecast_horizon=forecast_horizon,
    step=step,
    expanding=expanding,
    return_full_df=False
):

    train_series = train.copy()
    test_series = test.copy()

    # --- skip windows too small for lag construction ---
    if len(train_series) <= max_lag:
        continue

    # --- build lagged matrix ---
    X_train = []
    y_train = []

    for i in range(max_lag, len(train_series)):
        X_train.append(
            train_series.iloc[i-max_lag:i].values
        )
        y_train.append(
            train_series.iloc[i]
        )

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    # --- fit AR model ---
    model = LinearRegression()
    model.fit(X_train, y_train)

    # --- forecast next observation ---
    X_test = (
        train_series.iloc[-max_lag:]
        .values
        .reshape(1, -1)
    )

    forecast = model.predict(X_test)[0]
    actual = test_series.iloc[0]

    results.append({
        "date": test_series.index[0],
        "actual": actual,
        "forecast": forecast,
        "abs_error": abs(actual - forecast)
    })

# --- results dataframe ---
results_df = pd.DataFrame(results)

# --- compute MASE denominator ---
naive_errors = np.abs(
    track_B_df[target_col].diff().dropna()
)

mase_denominator = naive_errors.mean()

# --- compute MASE ---
mase = (
    results_df["abs_error"].mean()
    / mase_denominator
)

print(f"MASE: {mase:.4f}")

MASE: 3.3229


In [25]:
import numpy as np
import pandas as pd

# --- parameters ---
target_col = "UNRATE"
step = 5
forecast_horizon = 1
expanding = False

# --- storage ---
y_true_all = []
y_pred_all = []

# --- run POOS ---
for train, test in generate_poos_splits(
    track_B_df,
    target_col=target_col,
    sequence_length=1,   # not used for RW, but required by generator
    forecast_horizon=forecast_horizon,
    step=step,
    expanding=expanding
):
    train_values = train.values
    
    preds = []
    
    # Random walk: y_hat_t = y_{t-1}
    last_value = train_values[-1]
    
    for _ in range(len(test)):
        preds.append(last_value)
    
    # store predictions
    y_true_all.extend(test.values)
    y_pred_all.extend(preds)

# --- compute MASE ---
y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)

# model MAE
mae = np.mean(np.abs(y_true_all - y_pred_all))

# naive benchmark (lag-1 random walk denominator)
y_full = track_B_df[target_col].values
naive_mae = np.mean(np.abs(y_full[1:] - y_full[:-1]))

mase = mae / naive_mae

# --- output ---
print(f"Random Walk MASE: {mase:.4f}")

Random Walk MASE: 1.1976


In [26]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# --- parameters ---
target_col = "UNRATE"

L = 6                      # lag length
sequence_length = 60       # training window size

step = 5
forecast_horizon = 1
expanding = False

# --- storage ---
y_true_all = []
y_pred_all = []

# --- helper: create multivariate lag matrix ---
def create_var_lags(df, lags, target_col):

    X, y = [], []

    values = df.values
    target_idx = df.columns.get_loc(target_col)

    for i in range(lags, len(df)):

        # flatten lag window across all variables
        X.append(
            values[i-lags:i].flatten()
        )

        y.append(
            values[i, target_idx]
        )

    return np.array(X), np.array(y)

# --- run POOS ---
for train, test in generate_poos_splits(
    track_B_df,
    target_col=target_col,
    sequence_length=sequence_length,
    forecast_horizon=forecast_horizon,
    step=step,
    expanding=expanding,
    return_full_df=True
):

    train_df = train.copy()
    test_df = test.copy()

    # --- create lag matrix ---
    X_train, y_train = create_var_lags(
        train_df,
        L,
        target_col
    )

    # --- skip invalid windows ---
    if len(X_train) == 0:
        continue

    # --- fit model ---
    model = LinearRegression()
    model.fit(X_train, y_train)

    # --- recursive forecasting ---
    history = train_df.values.tolist()

    preds = []

    target_idx = train_df.columns.get_loc(target_col)

    for _ in range(len(test_df)):

        x_input = (
            np.array(history[-L:])
            .flatten()
            .reshape(1, -1)
        )

        pred = model.predict(x_input)[0]

        preds.append(pred)

        # update only target variable
        next_row = history[-1].copy()
        next_row[target_idx] = pred

        history.append(next_row)

    # --- store results ---
    y_true_all.extend(
        test_df[target_col].values
    )

    y_pred_all.extend(preds)

# --- compute MASE ---
y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)

mae = np.mean(
    np.abs(y_true_all - y_pred_all)
)

y_full = track_B_df[target_col].values

naive_mae = np.mean(
    np.abs(y_full[1:] - y_full[:-1])
)

mase = mae / naive_mae

print(f"VAR-style autoregression MASE: {mase:.4f}")

VAR-style autoregression MASE: 2.2914
